<a id="top"></a>

# Demo: the context window is one shared budget

```yaml
title: "Demo: the context window is one shared budget"
keywords: context window, tokens, budget, overflow, truncation, teacher demo
estimated duration: 15
```

> **Lesson:** L01. Teacher demo — Demo 2 in the [roadmap](../../../../docs/origin/lesson_roadmaps/L01/demos_or_activities.md). Anchor: Claude Sonnet 4.6, 200k window. Runs fully offline.

## Contents

- [1. Setup](#1-setup)
- [2. The shared budget](#2-the-shared-budget)
- [3. Three failure modes](#3-three-failure-modes)

## 1. Setup

In [1]:
WINDOW = 200_000  # Claude Sonnet 4.6 standard window, in tokens


def window_meter(
    system: int, history: int, current: int, reserved_output: int, width: int = 40
) -> str:
    """ASCII meter of how a 200k window is divided across the things competing for it."""
    used = system + history + current + reserved_output
    per_block = WINDOW / width
    filled = min(width, round(used / per_block))
    bar = "█" * filled + "░" * (width - filled)
    return f"[{bar}] {used:,}/{WINDOW:,} tokens used"


print("setup ready")

setup ready


[↑ Back to top](#top)

## 2. The shared budget

System message, conversation history, current input, and reserved output all draw from the *same* budget.

In [2]:
print("tiny call    ", window_meter(system=20, history=0, current=15, reserved_output=200))
print("with history ", window_meter(system=20, history=20 * 500, current=15, reserved_output=200))
print("overflow     ", window_meter(system=20, history=250_000, current=15, reserved_output=200))

overflow_total = 20 + 250_000 + 15 + 200
print(
    f"\noverflow total = {overflow_total:,} tokens > {WINDOW:,} window -> the API would HARD-REJECT this"
)

tiny call     [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░] 235/200,000 tokens used
with history  [██░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░] 10,235/200,000 tokens used
overflow      [████████████████████████████████████████] 250,235/200,000 tokens used

overflow total = 250,235 tokens > 200,000 window -> the API would HARD-REJECT this


[↑ Back to top](#top)

## 3. Three failure modes

Say these out loud while the overflow number is on screen:

1. **Hard rejection** — what the number above shows: the API refuses before running.
2. **Silent truncation** — some clients/frameworks quietly drop the oldest turns. The call
   succeeds but the model "forgot." *This is the dangerous one.*
3. **Quality degradation** — even when it fits, models can lose track of material buried
   mid-context. Foreshadows L17 (context management) and L19 (RAG).

[↑ Back to top](#top)